# RML pilot verification

Load the generated T1.3 Turtle artifact into the local `pilot-kg` Fuseki dataset and compare three SPARQL results with the source transaction CSV.

In [ ]:
import os
from decimal import Decimal
from pathlib import Path

import pandas as pd

from nl2sparql.kg.ontology.fuseki_pilot import FusekiPilotClient

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
TRANSACTIONS_PATH = ROOT / "data/raw/pilot/transactions_pilot.csv"
OUTPUT_PATH = ROOT / "data/processed/pilot/output.ttl"
assert TRANSACTIONS_PATH.is_file()
assert OUTPUT_PATH.is_file()

In [ ]:
transactions = pd.read_csv(TRANSACTIONS_PATH, dtype=str)
expected_transaction_count = len(transactions)
expected_high_value_count = sum(
    Decimal(value) > Decimal("1000000000000000000")
    for value in transactions["value"]
)
sender_counts = transactions["from_address"].value_counts().to_dict()
expected_top_senders = sorted(
    sender_counts.items(), key=lambda item: (-item[1], item[0])
)[:10]
assert expected_transaction_count == 100

In [ ]:
fuseki_url = os.environ.get("FUSEKI_URL", "http://localhost:3030")
fuseki_user = os.environ.get("FUSEKI_ADMIN_USER", "admin")
fuseki_password = os.environ.get("FUSEKI_ADMIN_PASSWORD", "admin")
client = FusekiPilotClient(
    fuseki_url,
    "pilot-kg",
    fuseki_user,
    fuseki_password,
)
client.prepare_dataset()
client.upload_turtle(OUTPUT_PATH)

In [ ]:
count_result = client.query("""
PREFIX : <https://thesis.example.org/eth-kg/>
SELECT (COUNT(?tx) AS ?n) WHERE { ?tx a :Transaction }
""")
actual_transaction_count = int(count_result["results"]["bindings"][0]["n"]["value"])
assert actual_transaction_count == expected_transaction_count

In [ ]:
high_value_result = client.query("""
PREFIX : <https://thesis.example.org/eth-kg/>
SELECT (COUNT(?tx) AS ?n) WHERE {
  ?tx a :Transaction ; :hasValue ?value .
  FILTER(?value > 1000000000000000000)
}
""")
actual_high_value_count = int(
    high_value_result["results"]["bindings"][0]["n"]["value"]
)
assert actual_high_value_count == expected_high_value_count

In [ ]:
top_sender_result = client.query("""
PREFIX : <https://thesis.example.org/eth-kg/>
SELECT ?from (COUNT(?tx) AS ?n_tx) WHERE {
  ?tx :hasFromAddress ?from .
}
GROUP BY ?from
ORDER BY DESC(?n_tx) ?from
LIMIT 10
""")
actual_top_senders = [
    (row["from"]["value"].rsplit("/", 1)[-1], int(row["n_tx"]["value"]))
    for row in top_sender_result["results"]["bindings"]
]
assert actual_top_senders == expected_top_senders